In [ ]:
import sys
from pathlib import Path

from bodge import Lattice
sys.path[:0] = [str(Path.cwd().parent)]

from concurrent.futures import ThreadPoolExecutor
import os

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

from scripts.constants import PI, s2  # type: ignore
from scripts.utils import is_hermitian

from scripts.Hamiltonian import Hamiltonian
from scripts.Lattice import Lattice
from scripts.self_consistency import bdg_sc

In [16]:
from random import seed


initial_seeds = np.array([
    [0,0,0,0],  # normal state
    [0.1, -0.1, 0, 0],  # px 
    [0, 0, 0.1, -0.1],  # py
    [0.1, -0.1, 0.1, -0.1],  # px + py
    [0.1, -0.1, 0.1j, -0.1j],  # px + i*py
    [0.1, 0.1, -0.1, -0.1],  # d-wave
    [0.1+0.1, 0.1-0.1, -0.1, -0.1],  # d-wave + px
    [0.1, 0.1, -0.1+0.1, -0.1-0.1],  # d-wave + py
    [0.1, 0.1, 0.1, 0.1],  # s-wave
    [0.1+0.1, 0.1-0.1, 0.1, 0.1],  # s-wave + px
    [0.1, 0.1, 0.1+0.1, 0.1-0.1] # s-wave + py
], dtype=np.complex128)

seed_strings = [ "normal state",
    "px", "py", "px+py", "px+i*py",
    "d-wave", "d-wave+px", "d-wave+py",
    "s-wave", "s-wave+px", "s-wave+py"
]


X, Y = 50, 50
lattice = Lattice(X, Y)

t = 1
mu = 0.8 * t * np.ones(lattice.X)
T = 1e-8

V0 = 1.5 * t / 2
V = V0 * np.ones(lattice.X)

for i, seed in enumerate(initial_seeds):
    H = Hamiltonian(t, mu, lattice, 
                    U=None, V_prime=None, V=V,
                    F_init=seed)
    F_swave, F_dwave, F_px, F_py = bdg_sc(H, temperature=T, verbose_free=False,
                                          rtol=0.01, atol=1e-5, maxiter=1000)
    print(f"F_swave: {F_swave[X//2]}, F_dwave: {F_dwave[X//2]}")
    print(f"F_px: {F_px[X//2]}, F_py: {F_py[X//2]}")
    print(f"Seed: {seed_strings[i]}, Free energy: {H.free}")

Converged after 1 iterations
F_swave: (1.3924721610283615e-18+0j), F_dwave: (9.539139835585703e-18+0j)
F_px: (2.3267447052126428e-17+0j), F_py: 1.2737270556056712e-17j
Seed: normal state, Free energy: -8871.175269619456
Converged after 114 iterations
F_swave: (0.0008518962258270424+0j), F_dwave: (-0.13569473762051867+0j)
F_px: (2.0822751630981804e-05+0j), F_py: -9.618094858487382e-17j
Seed: px, Free energy: -9038.592513343603
Converged after 223 iterations
F_swave: (-0.0008983536613511589+6.688503709665523e-10j), F_dwave: (0.13535068069913936-4.819432074269725e-08j)
F_px: (-3.5875816206676353e-16-1.1585320383400067e-17j), F_py: (0.015097962933837071-5.414612574439457e-09j)
Seed: py, Free energy: -9038.808544530486
Converged after 205 iterations
F_swave: (0.0008983519907212467-1.7065397614895072e-12j), F_dwave: (-0.13535070798464358+1.22965904056248e-10j)
F_px: (4.234460004859386e-15-2.491855858311788e-17j), F_py: (0.015097400678579639-1.3814748558925655e-11j)
Seed: px+py, Free energy: 

KeyboardInterrupt: 